# Solar Filament Segmentation — complete reproducible pipeline

This notebook contains the complete original source implementation. It starts
in **resume** mode: supply the released model bundle (tensor weights, config,
training manifest) and the official competition input, then run all cells.
Point `MODEL_DIR` at the released bundle if its mount location differs. No
training is required for inference. Training from scratch is an explicit mode.

The model uses only H-alpha pixels and training masks. All annotators of an
observation share one split: **558 train / 8 development / 141 held out**.
Only the fixed development images select among thresholds **0.45, 0.65, 0.80**.
The choice is frozen before the holdout or test images are evaluated.
The uncertainty interval uses 1,000 observation-cluster bootstrap resamples
(seed 2026) of already-recorded counts, keeping all annotators together. It
does not rerun inference. The 95% percentile interval is exploratory: this
filename-hash split does not establish temporal independence or coverage.

This is an unexecuted source notebook: it contains no fabricated scores,
predictions or weights. Running it generates local artifacts; it never uploads
the CSV or submits any form. Actual results belong in the generated manifests.

Data: [MAGFiLO competition](https://www.kaggle.com/competitions/filament-segmentation-2026/data),
NSO GONG H-alpha observations, CC BY-NC 4.0, noncommercial competition use.
The saved first-three holdout comparison panels are derived report figures,
not a redistribution of original JPEGs or annotation JSON.

[Organizer self-evaluation v6](https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook)
uses strict IoU > 0.5, all qualifying pairs, and global PQ across separate
annotator-image rows. Zero-GT annotation sets are excluded from its GT-row
universe. Images with no detections receive no invented CSV row; their names
are retained in the coverage manifest. Server acceptance of this empty-image
policy still requires an actual upload result.


## 1. Input paths and explicit execution mode


In [ ]:
from pathlib import Path
MODE = "resume"  # Explicitly change to "train" only to reproduce training.
DEVICE = "cuda"  # Use "cpu" for inference where CUDA is unavailable.
DATA_ROOT = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026")
MODEL_DIR = Path("model") if Path("model/config.json").is_file() else Path("/kaggle/working/unet-train558-model")
CHECKPOINT_DIR = Path("/kaggle/working/unet-train558-epoch")
OUTPUT_DIR = Path("/kaggle/working/solar-final-output")
DELIVERY_ZIP = Path("/kaggle/working/solar-final-delivery.zip")
DEVELOPMENT_REPORT = Path("outputs/development-results.json") if Path("outputs/development-results.json").is_file() else Path("/kaggle/working/solar-unet-train558-development.json")
ANNOTATION_PATH = DATA_ROOT / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
IMAGE_ROOT = DATA_ROOT / "train" / "train_images"
TEST_ROOT = DATA_ROOT / "test" / "test_images"
SOURCE_ROOT = Path("/kaggle/working/solar_final_source")
SOURCE_ROOT.mkdir(exist_ok=True)


## 2. Complete source
Each cell writes one readable reviewed module. No package installation, external weights or code download occurs.


### dataset.py
SHA-256: `0fb04691dfd78dbb121eb56362100a83616ca4c174169a3a0f18b3d07346321c`


In [ ]:
%%writefile /kaggle/working/solar_final_source/dataset.py
"""COCO polygon input and compressed-RLE output, without metadata features.

Requires actual pycocotools for rasterization/encoding. There is deliberately
no approximate OpenCV polygon fallback for evaluation ground truth.
"""
from __future__ import annotations

from dataclasses import dataclass
import json
from pathlib import Path
from typing import Any, Dict, List, Tuple

import cv2
import numpy as np


@dataclass(frozen=True)
class ImageRecord:
    image_id: str
    file_name: str
    width: int
    height: int


@dataclass
class CocoIndex:
    images: Dict[str, ImageRecord]
    observations: Dict[str, Tuple[str, ...]]
    segmentations: Dict[str, Tuple[Any, ...]]


def index_coco(payload: dict) -> CocoIndex:
    """Only id/file_name/shape and segmentation enter the index.

    file_name is the observation key, while image_id identifies an annotation
    set. Multiple image IDs for one filename must stay in the same split.
    URLs, dates, categories/spines and other annotation metadata are unused.
    """
    if not isinstance(payload, dict) or not isinstance(payload.get("images"), list) or not isinstance(payload.get("annotations"), list):
        raise ValueError("COCO payload needs images and annotations lists")
    images, observations, segmentations = {}, {}, {}
    for item in payload["images"]:
        image_id = item.get("id")
        file_name = item.get("file_name")
        width, height = item.get("width"), item.get("height")
        if not isinstance(image_id, str) or not image_id:
            raise ValueError("expected the actual dataset's nonempty string image id")
        if image_id in images:
            raise ValueError("duplicate image id: " + image_id)
        if not isinstance(file_name, str) or not file_name or "/" in file_name or "\\" in file_name or file_name in (".", ".."):
            raise ValueError("file_name must be a single safe image filename")
        if any(isinstance(v, bool) or not isinstance(v, int) or v <= 0 for v in (width, height)):
            raise ValueError("image dimensions must be positive integers")
        for previous_id in observations.get(file_name, ()):
            previous = images[previous_id]
            if (previous.width, previous.height) != (width, height):
                raise ValueError("annotators disagree on observation dimensions")
        images[image_id] = ImageRecord(image_id, file_name, width, height)
        observations.setdefault(file_name, []).append(image_id)
        segmentations[image_id] = []
    for annotation in payload["annotations"]:
        image_id = annotation.get("image_id")
        if image_id not in images:
            raise ValueError("annotation references an unknown image_id")
        segmentation = annotation.get("segmentation")
        if not isinstance(segmentation, (list, dict)) or not segmentation:
            raise ValueError("annotation needs a nonempty COCO polygon or RLE segmentation")
        if isinstance(segmentation, list):
            for polygon in segmentation:
                try:
                    coords = np.asarray(polygon, dtype=np.float64)
                except (ValueError, TypeError) as error:
                    raise ValueError("invalid polygon coordinates") from error
                if coords.ndim != 1 or coords.size < 6 or coords.size % 2 or not np.isfinite(coords).all():
                    raise ValueError("polygon must contain >=3 finite x/y pairs")
        segmentations[image_id].append(segmentation)
    return CocoIndex(images, {k: tuple(v) for k, v in observations.items()},
                     {k: tuple(v) for k, v in segmentations.items()})


def load_coco_index(path) -> CocoIndex:
    with Path(path).open("r", encoding="utf-8") as stream:
        return index_coco(json.load(stream))


def _mask_api():
    try:
        from pycocotools import mask
    except ImportError as error:
        raise RuntimeError("Official COCO polygon/RLE conversion requires pycocotools; no approximate ground-truth fallback is enabled") from error
    return mask


def ground_truth_masks(index: CocoIndex, image_id: str) -> List[np.ndarray]:
    api = _mask_api()
    record = index.images[image_id]
    result = []
    for segmentation in index.segmentations[image_id]:
        if isinstance(segmentation, list):
            rle = api.merge(api.frPyObjects(segmentation, record.height, record.width))
        elif isinstance(segmentation.get("counts"), list):
            rle = api.frPyObjects(segmentation, record.height, record.width)
        else:
            rle = dict(segmentation)
            if isinstance(rle.get("counts"), str):
                rle["counts"] = rle["counts"].encode("ascii")
        decoded = np.asarray(api.decode(rle))
        if decoded.shape != (record.height, record.width):
            raise ValueError("decoded annotation mask shape mismatch")
        binary = decoded.astype(bool)
        if not binary.any():
            raise ValueError("decoded annotation is an empty instance")
        result.append(binary)
    return result


def read_observation(index: CocoIndex, image_root, observation: str) -> np.ndarray:
    image_root = Path(image_root).resolve()
    record = index.images[index.observations[observation][0]]
    path = (image_root / record.file_name).resolve()
    if path.parent != image_root:
        raise ValueError("image path escapes the supplied image directory")
    image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if image is None:
        raise ValueError("could not read observation: " + record.file_name)
    if image.shape != (record.height, record.width) or image.dtype != np.uint8:
        raise ValueError("actual image must match annotation dimensions and uint8 grayscale")
    return image


def submission_rows(file_name: str, masks, expected_shape=(2048, 2048), first_instance=1) -> List[dict]:
    """One compressed counts string per nonempty instance; no data upload.

    first_instance controls only the unique numeric suffix. Confirm the
    official sample's suffix convention before a real final submission.
    """
    api = _mask_api()
    stem = Path(file_name).stem
    if not stem or isinstance(first_instance, bool) or not isinstance(first_instance, int) or first_instance < 0:
        raise ValueError("invalid image stem or first_instance")
    rows = []
    for number, mask in enumerate(masks, start=first_instance):
        mask = np.asarray(mask)
        if mask.shape != tuple(expected_shape) or mask.dtype.kind not in "biuf" or not np.isfinite(mask).all() or not np.all((mask == 0) | (mask == 1)) or not np.any(mask):
            raise ValueError("submission masks must be nonempty binary masks of the official shape")
        rle = api.encode(np.asfortranarray(mask, dtype=np.uint8))
        counts = rle["counts"]
        if isinstance(counts, bytes):
            counts = counts.decode("ascii")
        if not isinstance(counts, str):
            raise ValueError("expected pycocotools compressed RLE counts")
        if not np.array_equal(api.decode(rle).astype(bool), mask.astype(bool)):
            raise ValueError("RLE round-trip mismatch")
        rows.append({"filament_id": stem + "_" + str(number), "segmentation_rle": counts})
    return rows


### evaluation.py
SHA-256: `4548a68482a2401fd6ee4b73a51f21eef07f8ae477551c3b4d9d8a4f3b64df67`


In [ ]:
%%writefile /kaggle/working/solar_final_source/evaluation.py
"""Original mask-only PQ and observation-group validation utilities.

The pair score follows the organizer rule supplied for this project: every
pair with IoU strictly greater than 0.5 is a hit. No Hungarian matching is
performed. FP/FN are instances without any hit. The organizer self-eval v6
returns 0.0 when the denominator is zero and computes the final score from
counts accumulated over all annotator-image comparisons.

COCO polygon rasterization belongs in the data adapter, not this module.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
import hashlib
import math
from numbers import Integral, Real
from typing import Iterable, Mapping, Optional, Sequence, Tuple

import numpy as np


@dataclass(frozen=True)
class PQResult:
    pq: float
    sum_iou: float
    tp: int
    fp: int
    fn: int

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass(frozen=True)
class ObservationSplit:
    train: Tuple[str, ...]
    validation: Tuple[str, ...]

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass(frozen=True)
class AnnotationEvaluation:
    # Official primary result: aggregate counts before taking the ratio.
    official_global_pq: float
    global_counts: Mapping[str, object]
    # Diagnostic only: mean over individual annotator-image comparisons.
    macro_pq: float
    # Diagnostic only: equal-observation weighting.
    observation_macro_pq: float
    comparison_count: int
    by_observation: Mapping[str, Mapping[str, PQResult]]

    def to_dict(self) -> dict:
        return {
            "official_global_pq": self.official_global_pq,
            "global_counts": dict(self.global_counts),
            "macro_pq": self.macro_pq,
            "observation_macro_pq": self.observation_macro_pq,
            "comparison_count": self.comparison_count,
            "by_observation": {
                observation: {
                    annotator: result.to_dict()
                    for annotator, result in comparisons.items()
                }
                for observation, comparisons in self.by_observation.items()
            },
        }


def _shape(shape: Optional[Sequence[int]]) -> Optional[Tuple[int, int]]:
    if shape is None:
        return None
    try:
        values = tuple(shape)
    except TypeError as error:
        raise ValueError("expected_shape must contain two positive integers") from error
    if len(values) != 2 or any(
        isinstance(value, (bool, np.bool_))
        or not isinstance(value, Integral)
        or value <= 0
        for value in values
    ):
        raise ValueError("expected_shape must contain two positive integers")
    return int(values[0]), int(values[1])


def _masks(
    masks: Iterable[np.ndarray],
    expected_shape: Optional[Tuple[int, int]],
    label: str,
) -> Tuple[Tuple[np.ndarray, ...], Optional[Tuple[int, int]]]:
    if isinstance(masks, np.ndarray) and masks.ndim != 3:
        raise ValueError(label + " must be a collection of 2D masks, not one 2D mask")
    try:
        values = tuple(masks)
    except TypeError as error:
        raise ValueError(label + " must be an iterable of 2D masks") from error
    result = []
    shape = expected_shape
    for index, value in enumerate(values):
        mask = np.asarray(value)
        name = "{}[{}]".format(label, index)
        if mask.ndim != 2 or any(dimension <= 0 for dimension in mask.shape):
            raise ValueError(name + " must be a nonempty 2D array")
        if shape is None:
            shape = tuple(mask.shape)
        if tuple(mask.shape) != shape:
            raise ValueError(
                "{} shape {} does not match {}".format(name, mask.shape, shape)
            )
        if mask.dtype.kind not in "biuf":
            raise ValueError(name + " must contain binary numeric or boolean values")
        if not np.all(np.isfinite(mask)):
            raise ValueError(name + " contains a nonfinite value")
        if not np.all((mask == 0) | (mask == 1)):
            raise ValueError(name + " must contain only 0 and 1")
        binary = mask.astype(bool, copy=False)
        if not np.any(binary):
            raise ValueError(name + " is an empty instance; omit it instead")
        result.append(binary)
    return tuple(result), shape


def _bounds(mask: np.ndarray) -> Tuple[int, int, int, int]:
    rows = np.flatnonzero(np.any(mask, axis=1))
    columns = np.flatnonzero(np.any(mask, axis=0))
    return int(rows[0]), int(rows[-1] + 1), int(columns[0]), int(columns[-1] + 1)


def instance_pq(
    pred_masks: Iterable[np.ndarray],
    gt_masks: Iterable[np.ndarray],
    expected_shape: Optional[Sequence[int]] = None,
) -> PQResult:
    """Calculate strict IoU > 0.5 all-hit instance PQ.

    Masks must be 2D binary arrays of identical shape; an instance with zero
    pixels is invalid. Pass [] for no instances. Lists, generators, and
    (n, height, width) NumPy arrays are accepted. A single 2D array must be
    explicitly wrapped in a list.

    Every qualifying pair contributes to tp and sum_iou. An instance is
    unmatched only if no pair involving it qualifies. This deliberately does
    not impose one-to-one assignment, even for duplicated/overlapping masks.
    """
    target_shape = _shape(expected_shape)
    predictions, target_shape = _masks(pred_masks, target_shape, "pred_masks")
    ground_truth, target_shape = _masks(gt_masks, target_shape, "gt_masks")
    pred_area = [int(np.count_nonzero(mask)) for mask in predictions]
    gt_area = [int(np.count_nonzero(mask)) for mask in ground_truth]
    pred_bounds = [_bounds(mask) for mask in predictions]
    gt_bounds = [_bounds(mask) for mask in ground_truth]
    pred_hit = np.zeros(len(predictions), dtype=bool)
    gt_hit = np.zeros(len(ground_truth), dtype=bool)
    matched_ious = []
    for pred_index, prediction in enumerate(predictions):
        py0, py1, px0, px1 = pred_bounds[pred_index]
        for gt_index, target in enumerate(ground_truth):
            gy0, gy1, gx0, gx1 = gt_bounds[gt_index]
            y0, y1 = max(py0, gy0), min(py1, gy1)
            x0, x1 = max(px0, gx0), min(px1, gx1)
            if y0 >= y1 or x0 >= x1:
                continue
            intersection = int(
                np.count_nonzero(
                    prediction[y0:y1, x0:x1] & target[y0:y1, x0:x1]
                )
            )
            union = pred_area[pred_index] + gt_area[gt_index] - intersection
            # Compare integers for the strict threshold, avoiding rounding a
            # true boundary IoU=0.5 above the cutoff.
            if 2 * intersection > union:
                matched_ious.append(intersection / union)
                pred_hit[pred_index] = True
                gt_hit[gt_index] = True
    true_positives = len(matched_ious)
    false_positives = int(len(predictions) - np.count_nonzero(pred_hit))
    false_negatives = int(len(ground_truth) - np.count_nonzero(gt_hit))
    sum_iou = math.fsum(matched_ious)
    denominator = true_positives + 0.5 * false_positives + 0.5 * false_negatives
    pq = sum_iou / denominator if denominator else 0.0
    return PQResult(
        pq=float(pq),
        sum_iou=float(sum_iou),
        tp=true_positives,
        fp=false_positives,
        fn=false_negatives,
    )


def group_by_observation_split(
    observation_ids: Iterable[str],
    validation_fraction: float = 0.2,
    seed: int = 2026,
) -> ObservationSplit:
    """Split unique observations, never individual annotator/image IDs.

    Supply the shared physical-observation key (the image file_name in the
    current dataset), repeated as needed for its annotators. Hash ordering
    makes the split independent of input order and Python's randomized hash.
    At least two distinct observations are required. Both subsets are kept
    nonempty; therefore very small groups may not realize the exact fraction.
    """
    if (
        isinstance(validation_fraction, (bool, np.bool_))
        or not isinstance(validation_fraction, Real)
        or not math.isfinite(validation_fraction)
        or not 0.0 < validation_fraction < 1.0
    ):
        raise ValueError("validation_fraction must be finite and between 0 and 1")
    if isinstance(seed, (bool, np.bool_)) or not isinstance(seed, Integral):
        raise ValueError("seed must be an integer")
    ids = tuple(observation_ids)
    if any(not isinstance(value, str) or not value.strip() for value in ids):
        raise ValueError("observation IDs must be nonempty strings")
    unique_ids = set(ids)
    if len(unique_ids) < 2:
        raise ValueError("at least two distinct observations are required")
    ordered = sorted(
        unique_ids,
        key=lambda value: (
            hashlib.sha256((str(int(seed)) + "\0" + value).encode("utf-8")).digest(),
            value,
        ),
    )
    validation_count = max(
        1, min(len(ordered) - 1, int(round(len(ordered) * validation_fraction)))
    )
    return ObservationSplit(
        train=tuple(sorted(ordered[validation_count:])),
        validation=tuple(sorted(ordered[:validation_count])),
    )


def evaluate_annotators(
    predictions_by_observation: Mapping[str, Iterable[np.ndarray]],
    annotations_by_observation: Mapping[
        str, Mapping[str, Iterable[np.ndarray]]
    ],
    expected_shape: Optional[Sequence[int]] = None,
) -> AnnotationEvaluation:
    """Compare one prediction set separately against every annotator.

    The nested annotation mapping is observation_key -> annotator_or_image_id
    -> masks. Do not combine different annotators' instance masks. Missing
    predictions for a known observation mean no predicted instances; an
    unknown prediction observation is rejected as a probable keying error.

    official_global_pq follows organizer self-eval v6: sum IoUs, TP, FP and
    FN over every annotator-image comparison, then calculate one PQ ratio.
    A zero denominator returns 0.0. The two macro averages are diagnostics
    only: macro_pq weights each comparison equally; observation_macro_pq
    averages annotators first and then observations.
    """
    if not annotations_by_observation:
        raise ValueError("at least one annotated observation is required")
    extras = set(predictions_by_observation) - set(annotations_by_observation)
    if extras:
        raise ValueError("predictions include unannotated observations: " + repr(sorted(extras)))
    target_shape = _shape(expected_shape)
    by_observation = {}
    all_scores = []
    observation_scores = []
    all_results = []
    for observation in sorted(annotations_by_observation):
        annotators = annotations_by_observation[observation]
        if not annotators:
            raise ValueError("each observation requires at least one annotator")
        # Materialize once so generators can be reused for every annotator.
        predictions = tuple(predictions_by_observation.get(observation, ()))
        results = {}
        for annotator in sorted(annotators):
            result = instance_pq(
                predictions, annotators[annotator], expected_shape=target_shape
            )
            results[annotator] = result
            all_scores.append(result.pq)
            all_results.append(result)
        by_observation[observation] = results
        observation_scores.append(math.fsum(r.pq for r in results.values()) / len(results))
    global_sum_iou = math.fsum(result.sum_iou for result in all_results)
    global_tp = sum(result.tp for result in all_results)
    global_fp = sum(result.fp for result in all_results)
    global_fn = sum(result.fn for result in all_results)
    denominator = global_tp + 0.5 * global_fp + 0.5 * global_fn
    return AnnotationEvaluation(
        official_global_pq=global_sum_iou / denominator if denominator else 0.0,
        global_counts={
            "sum_iou": global_sum_iou,
            "tp": global_tp,
            "fp": global_fp,
            "fn": global_fn,
        },
        macro_pq=math.fsum(all_scores) / len(all_scores),
        observation_macro_pq=math.fsum(observation_scores) / len(observation_scores),
        comparison_count=len(all_scores),
        by_observation=by_observation,
    )



### unet_candidate.py
SHA-256: `04de330d8b304775487ea35216d87ee422391f5b8a4eca972dd15aa032cc8fe0`


In [ ]:
%%writefile /kaggle/working/solar_final_source/unet_candidate.py
"""Original small U-Net candidate, trained from scratch on explicit train names.

No pretrained weights, downloads, network access or automatic data discovery.
Training unions annotators only for the semantic target; official instance PQ
must still evaluate each original annotator separately at the original size.
No real-data quality is established by this module or its synthetic tests.
"""
from __future__ import annotations

from collections import OrderedDict
from dataclasses import asdict, dataclass
import json
import math
from pathlib import Path
import time

import cv2
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from scipy import ndimage


@dataclass(frozen=True)
class UNetConfig:
    input_size: int = 512
    channels: tuple = (16, 32, 64, 128)
    epochs: int = 8
    batch_size: int = 4
    learning_rate: float = 0.001
    positive_weight: float = 12.0
    random_state: int = 2026
    cache_bytes: int = 256 * 1024 * 1024
    probability_threshold: float = 0.65
    min_area: int = 128
    max_area_fraction: float = 0.05
    min_elongation: float = 1.2
    disk_threshold: int = 12
    limb_margin: int = 16
    max_instances: int = 256
    augment: bool = True
    device: str = "cpu"

    def __post_init__(self):
        bounds = {
            "input_size": (32, 1024), "epochs": (1, 100),
            "batch_size": (1, 16), "random_state": (0, 2**32 - 1),
            "cache_bytes": (0, 256 * 1024 * 1024), "min_area": (1, 2**31 - 1),
            "disk_threshold": (1, 255), "limb_margin": (0, 2048),
            "max_instances": (1, 4096),
        }
        for name, (low, high) in bounds.items():
            value = getattr(self, name)
            if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or not low <= value <= high:
                raise ValueError(name + " outside its bounded integer range")
        if self.input_size % 8:
            raise ValueError("input_size must be divisible by eight")
        if not isinstance(self.channels, tuple) or len(self.channels) != 4 or any(
            isinstance(c, bool) or not isinstance(c, int) or c < 4 or c > 128 or c % 4
            for c in self.channels
        ):
            raise ValueError("channels must be four multiples of four in 4..128")
        ranges = {
            "learning_rate": (0, 0.1), "positive_weight": (0, 100),
            "probability_threshold": (0, 1), "max_area_fraction": (0, 1),
        }
        for name, (low, high) in ranges.items():
            value = getattr(self, name)
            if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or not low < value <= high:
                raise ValueError(name + " must be finite and within bounds")
        if not isinstance(self.min_elongation, (int, float)) or isinstance(self.min_elongation, bool) or not math.isfinite(self.min_elongation) or self.min_elongation < 1:
            raise ValueError("min_elongation must be finite and at least one")
        if self.device not in ("cpu", "cuda"):
            raise ValueError("device must explicitly be cpu or cuda")
        if not isinstance(self.augment, bool):
            raise ValueError("augment must be bool")

    def to_dict(self):
        return asdict(self)


def _image(image):
    image = np.asarray(image)
    if image.ndim != 2 or not image.size or image.dtype != np.uint8:
        raise ValueError("image must be a nonempty uint8 grayscale array")
    return np.ascontiguousarray(image)


def _target(mask, shape):
    mask = np.asarray(mask)
    if mask.shape != shape or mask.dtype.kind not in "biuf" or not np.isfinite(mask).all() or not np.all((mask == 0) | (mask == 1)):
        raise ValueError("target must be a shape-matched binary union mask")
    return mask.astype(np.uint8)


def prepare_pair(image, mask, input_size):
    """Return compact uint8 image and quantized foreground coverage target.

    Area-resampled masks preserve partial coverage of thin filaments instead
    of silently dropping them with nearest-neighbour downsampling.
    """
    image = _image(image)
    target = _target(mask, image.shape)
    image_small = cv2.resize(image, (input_size, input_size), interpolation=cv2.INTER_AREA)
    coverage = cv2.resize(target.astype(np.float32), (input_size, input_size), interpolation=cv2.INTER_AREA)
    return image_small, np.rint(np.clip(coverage, 0, 1) * 255).astype(np.uint8)


def image_channels(image_small):
    """Two image-only channels: intensity and signed local normalized contrast."""
    gray = image_small.astype(np.float32) / np.float32(255.0)
    mean = cv2.GaussianBlur(gray, (49, 49), sigmaX=8.0, borderType=cv2.BORDER_REFLECT_101)
    contrast = np.clip((mean - gray) / np.maximum(mean, np.float32(0.05)), -1, 1)
    return np.stack((gray, contrast), axis=0).astype(np.float32)


class _ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(4, out_channels), nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(4, out_channels), nn.SiLU(),
        )

    def forward(self, image):
        return self.layers(image)


class SmallUNet(nn.Module):
    def __init__(self, channels=(16, 32, 64, 128)):
        super().__init__()
        a, b, c, d = channels
        self.encoder = nn.ModuleList((
            _ConvBlock(2, a), _ConvBlock(a, b), _ConvBlock(b, c), _ConvBlock(c, d),
        ))
        self.decoder = nn.ModuleList((
            _ConvBlock(d + c, c), _ConvBlock(c + b, b), _ConvBlock(b + a, a),
        ))
        self.head = nn.Conv2d(a, 1, 1)

    def forward(self, image):
        skips = []
        value = image
        for number, layer in enumerate(self.encoder):
            if number:
                value = F.max_pool2d(value, 2)
            value = layer(value)
            skips.append(value)
        for layer, skip in zip(self.decoder, reversed(skips[:-1])):
            value = F.interpolate(value, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            value = layer(torch.cat((value, skip), dim=1))
        return self.head(value)


def segmentation_loss(logits, target, positive_weight=12.0):
    """BCE-with-logits plus soft Dice, both averaged over the batch."""
    positive_weight = torch.as_tensor(positive_weight, dtype=logits.dtype, device=logits.device)
    bce = F.binary_cross_entropy_with_logits(logits, target, pos_weight=positive_weight)
    scores = torch.sigmoid(logits)
    axes = (1, 2, 3)
    dice = (2 * (scores * target).sum(axes) + 1) / (scores.sum(axes) + target.sum(axes) + 1)
    return bce + (1 - dice).mean()


class _PairCache:
    """Bounded LRU of compressed uint8 image/coverage pairs, not tensors."""
    def __init__(self, max_bytes):
        self.max_bytes = max_bytes
        self.bytes = 0
        self.peak_bytes = 0
        self.values = OrderedDict()

    def get(self, name, loader):
        if name in self.values:
            self.values.move_to_end(name)
            return self.values[name]
        value = loader(name)
        size = sum(array.nbytes for array in value)
        if size <= self.max_bytes:
            while self.values and self.bytes + size > self.max_bytes:
                _, removed = self.values.popitem(last=False)
                self.bytes -= sum(array.nbytes for array in removed)
            self.values[name] = value
            self.bytes += size
            self.peak_bytes = max(self.peak_bytes, self.bytes)
        return value


def _training_names(observations, allowed_training_observations, forbidden_observations):
    names = tuple(observations)
    if not names or any(not isinstance(n, str) or not n or "/" in n or "\\" in n for n in names):
        raise ValueError("explicit nonempty observation filename list required")
    if len(set(names)) != len(names):
        raise ValueError("duplicate training observation")
    if allowed_training_observations is None:
        raise ValueError("allowed_training_observations must explicitly establish the train split")
    allowed, forbidden = set(allowed_training_observations), set(forbidden_observations)
    if not set(names) <= allowed or set(names) & forbidden:
        raise ValueError("training observations cross the declared split")
    return names


class UNetCandidate:
    def __init__(self, config=None):
        self.config = config or UNetConfig()
        if not isinstance(self.config, UNetConfig):
            raise TypeError("config must be UNetConfig")
        self.model_ = None
        self.training_summary_ = None

    def _device(self):
        if self.config.device == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but unavailable; no silent CPU fallback")
        return torch.device(self.config.device)

    def _fit_loader(self, names, loader, progress=None, checkpoint_dir=None):
        config = self.config
        self.model_, self.training_summary_ = None, None
        device = self._device()
        torch.manual_seed(config.random_state)
        if device.type == "cuda":
            torch.cuda.manual_seed_all(config.random_state)
        rng = np.random.default_rng(config.random_state)
        cache = _PairCache(config.cache_bytes)
        model = SmallUNet(config.channels).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=1e-4)
        history = []
        started = time.perf_counter()
        model.train()
        for epoch in range(config.epochs):
            shuffled = rng.permutation(len(names))
            total_loss, total_examples = 0.0, 0
            for start in range(0, len(names), config.batch_size):
                inputs, targets = [], []
                for number in shuffled[start:start + config.batch_size]:
                    gray, coverage = cache.get(names[int(number)], loader)
                    # Copies ensure augmentation never mutates cached arrays.
                    gray, coverage = gray.copy(), coverage.copy()
                    if config.augment:
                        rotation = int(rng.integers(0, 4))
                        gray, coverage = np.rot90(gray, rotation), np.rot90(coverage, rotation)
                        if rng.random() < 0.5:
                            gray, coverage = np.fliplr(gray), np.fliplr(coverage)
                    inputs.append(image_channels(np.ascontiguousarray(gray)))
                    targets.append(np.ascontiguousarray(coverage[None], dtype=np.float32) / 255.0)
                batch = torch.from_numpy(np.stack(inputs)).to(device)
                target = torch.from_numpy(np.stack(targets)).to(device)
                optimizer.zero_grad(set_to_none=True)
                loss = segmentation_loss(model(batch), target, config.positive_weight)
                if not torch.isfinite(loss):
                    raise RuntimeError("nonfinite training loss")
                loss.backward()
                optimizer.step()
                total_loss += float(loss.detach().cpu()) * len(inputs)
                total_examples += len(inputs)
            item = {"epoch": epoch + 1, "mean_training_loss": total_loss / total_examples,
                    "elapsed_seconds": time.perf_counter() - started}
            history.append(item)
            if checkpoint_dir is not None:
                manifest = {"status": "epoch_checkpoint_not_heldout_evaluated",
                            "observations": list(names), "observation_count": len(names),
                            "epochs_completed": epoch + 1, "epochs_requested": config.epochs,
                            "history": history, "external_weights": False,
                            "torch_version": str(torch.__version__), "device": str(device),
                            "cache_peak_bytes": cache.peak_bytes}
                self._save_bundle(checkpoint_dir, model, manifest)
            if progress is not None:
                progress(dict(item))
        model.eval()
        self.model_ = model
        self.training_summary_ = {
            "observations": list(names), "observation_count": len(names),
            "epochs": config.epochs, "history": history,
            "parameter_count": sum(p.numel() for p in model.parameters()),
            "cache_peak_bytes": cache.peak_bytes, "cache_limit_bytes": config.cache_bytes,
            "device": str(device), "torch_version": str(torch.__version__),
            "external_weights": False, "training_targets": "union of training annotators only",
            "label_resize": "area coverage quantized to uint8/255",
            "quality_claim": "training loss is not held-out instance PQ",
        }
        return self

    def fit_coco(self, index, image_root, observations, *,
                 allowed_training_observations, forbidden_observations=(), progress=None,
                 checkpoint_dir=None):
        """Train only on explicit, disjoint observation names from split.train."""
        from dataset import ground_truth_masks, read_observation
        names = _training_names(observations, allowed_training_observations, forbidden_observations)
        if not set(names) <= set(index.observations):
            raise ValueError("unknown training observation")

        def load(name):
            image = read_observation(index, image_root, name)
            union = np.zeros(image.shape, dtype=bool)
            for image_id in index.observations[name]:
                for mask in ground_truth_masks(index, image_id):
                    union |= mask
            return prepare_pair(image, union, self.config.input_size)
        return self._fit_loader(names, load, progress, checkpoint_dir)

    def fit_pairs(self, observations, *, allowed_training_observations,
                  forbidden_observations=(), progress=None, checkpoint_dir=None):
        """Small in-memory/synthetic adapter mapping name -> (image, union)."""
        names = _training_names(tuple(observations), allowed_training_observations, forbidden_observations)
        def load(name):
            image, union = observations[name]
            return prepare_pair(image, union, self.config.input_size)
        return self._fit_loader(names, load, progress, checkpoint_dir)

    def predict_proba(self, image):
        if self.model_ is None:
            raise RuntimeError("fit or load_weights must complete before inference")
        image = _image(image)
        gray = cv2.resize(image, (self.config.input_size, self.config.input_size), interpolation=cv2.INTER_AREA)
        batch = torch.from_numpy(image_channels(gray)[None]).to(self._device())
        self.model_.eval()
        with torch.inference_mode():
            prediction = torch.sigmoid(self.model_(batch))
            prediction = F.interpolate(prediction, size=image.shape, mode="bilinear", align_corners=False)
            probability = prediction[0, 0].cpu().numpy().astype(np.float32)
        if probability.shape != image.shape or not np.isfinite(probability).all():
            raise RuntimeError("invalid model probability output")
        return np.clip(probability, 0, 1)

    def labels_from_probability(self, image, probabilities, probability_threshold=None):
        """Shared, bounded connected-component postprocessing at original size."""
        image = _image(image)
        probabilities = np.asarray(probabilities)
        if probabilities.shape != image.shape or probabilities.dtype.kind not in "biuf" or not np.isfinite(probabilities).all() or np.any(probabilities < 0) or np.any(probabilities > 1):
            raise ValueError("probabilities must match image shape and lie in [0,1]")
        threshold = self.config.probability_threshold if probability_threshold is None else probability_threshold
        if isinstance(threshold, bool) or not isinstance(threshold, (int, float)) or not math.isfinite(threshold) or not 0 < threshold <= 1:
            raise ValueError("threshold must be in (0,1]")
        config = self.config
        disk_labels, count = ndimage.label(image >= config.disk_threshold)
        output = np.zeros(image.shape, dtype=np.int32)
        if not count:
            return output
        sizes = np.bincount(disk_labels.ravel())
        sizes[0] = 0
        disk = ndimage.binary_fill_holes(disk_labels == int(np.argmax(sizes)))
        if config.limb_margin:
            distance = cv2.distanceTransform(np.pad(disk.astype(np.uint8), 1), cv2.DIST_L2, 5)[1:-1, 1:-1]
            disk = distance > config.limb_margin
        count, raw_labels, stats, _ = cv2.connectedComponentsWithStats(
            (disk & (probabilities >= threshold)).astype(np.uint8), connectivity=8, ltype=cv2.CV_32S)
        maximum = config.max_area_fraction * int(np.count_nonzero(disk))
        kept = 0
        for raw_id in range(1, count):
            left, top, width, height, area = map(int, stats[raw_id])
            if area < config.min_area or area > maximum:
                continue
            region = np.s_[top:top + height, left:left + width]
            local = raw_labels[region] == raw_id
            yy, xx = np.nonzero(local)
            coordinates = np.column_stack((xx, yy)).astype(np.float64)
            coordinates -= coordinates.mean(axis=0)
            eigenvalues = np.linalg.eigvalsh(coordinates.T @ coordinates / area)
            elongation = math.sqrt((eigenvalues[-1] + 1) / (eigenvalues[0] + 1))
            if elongation < config.min_elongation:
                continue
            kept += 1
            if kept > config.max_instances:
                raise RuntimeError("instance safety cap exceeded; no predictions silently truncated")
            output[region][local] = kept
        return output

    def predict_labels(self, image, probability_threshold=None):
        return self.labels_from_probability(image, self.predict_proba(image), probability_threshold)

    def predict_instances(self, image, probability_threshold=None):
        labels = self.predict_labels(image, probability_threshold)
        return [labels == label for label in range(1, int(labels.max()) + 1)]

    def save_weights(self, path):
        """Save state_dict tensors only; keep config and source alongside it."""
        if self.model_ is None:
            raise RuntimeError("fit must complete before saving")
        state = {name: value.detach().cpu() for name, value in self.model_.state_dict().items()}
        torch.save(state, Path(path))

    def _save_bundle(self, directory, model, manifest):
        directory = Path(directory)
        directory.mkdir(parents=True, exist_ok=True)
        state = {name: value.detach().cpu() for name, value in model.state_dict().items()}
        torch.save(state, directory / "weights.pt")
        (directory / "config.json").write_text(json.dumps(self.config.to_dict(), indent=2) + "\n", encoding="utf-8")
        (directory / "training-manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

    def save_artifacts(self, directory):
        """Private/local bundle: tensor state, config, and actual train manifest."""
        if self.model_ is None:
            raise RuntimeError("fit must complete before saving")
        self._save_bundle(directory, self.model_, self.training_summary_)

    @classmethod
    def load_artifacts(cls, directory, *, device=None):
        directory = Path(directory)
        config = json.loads((directory / "config.json").read_text(encoding="utf-8"))
        config["channels"] = tuple(config["channels"])
        if device is not None:
            config["device"] = device
        predictor = cls(UNetConfig(**config)).load_weights(directory / "weights.pt")
        predictor.training_summary_ = json.loads((directory / "training-manifest.json").read_text(encoding="utf-8"))
        return predictor

    def load_weights(self, path):
        """Load only a reviewed local state_dict, never arbitrary pickle objects."""
        model = SmallUNet(self.config.channels).to(self._device())
        model.load_state_dict(torch.load(Path(path), map_location=self._device(), weights_only=True), strict=True)
        self.model_ = model.eval()
        self.training_summary_ = {"loaded_local_state_dict": True, "training_history_available": False}
        return self


### pipeline.py
SHA-256: `63527af6fa04e722e0e7fd6af22f8eb244b819722a87ac607250189f9e5b1996`


In [ ]:
%%writefile /kaggle/working/solar_final_source/pipeline.py
"""Reproducible Solar post-training pipeline. No downloads or uploads.

Dev-only threshold freeze precedes the reserved holdout and test inference.
Only genuine pycocotools is used for native compressed-COCO verification.
The server's acceptance of zero-row images remains unverified by this code.
"""
from __future__ import annotations

import csv
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import math
from pathlib import Path
import platform
import sys
import zipfile

import cv2
import numpy as np

from dataset import ground_truth_masks, load_coco_index, read_observation, submission_rows
from evaluation import group_by_observation_split, instance_pq
from unet_candidate import UNetCandidate, UNetConfig

THRESHOLDS = (0.45, 0.65, 0.80)
OFFICIAL_SHAPE = (2048, 2048)
EXPECTED_COUNTS = (707, 566, 141, 558, 8, 180)
EMPTY_POLICY = "No CSV row for an image with zero detections; record it in coverage manifest. Offline evaluator supports this; server acceptance unverified."


def _now():
    return datetime.now(timezone.utc).isoformat()


def _hash_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for part in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(part)
    return digest.hexdigest()


def _canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode("utf-8")


def _digest(value):
    return hashlib.sha256(_canonical(value)).hexdigest()


def _json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def _read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def _counts():
    return dict(sum_iou=0.0, tp=0, fp=0, fn=0)


def _add_counts(total, score):
    for key in total:
        total[key] += score[key] if isinstance(score, dict) else getattr(score, key)


def _pq(counts):
    denominator = counts["tp"] + 0.5 * counts["fp"] + 0.5 * counts["fn"]
    return counts["sum_iou"] / denominator if denominator else 0.0


def partitions(index, strict=True):
    """Same observation grouping and evenly spaced dev8 as actual training."""
    split = group_by_observation_split([row.file_name for row in index.images.values()], 0.2, 2026)
    development = [split.train[int(i)] for i in np.linspace(0, len(split.train) - 1, 8)]
    training = [name for name in split.train if name not in set(development)]
    heldout = list(split.validation)
    if len(development) != len(set(development)):
        raise ValueError("not enough observations for distinct dev8")
    if set(training) & set(development) or set(training + development) & set(heldout):
        raise ValueError("observation split leakage")
    actual = (len(index.observations), len(split.train), len(heldout), len(training), len(development))
    if strict and actual != EXPECTED_COUNTS[:5]:
        raise ValueError("unexpected official training counts: " + repr(actual))
    return dict(training=training, development=development, holdout=heldout,
                combined_train=list(split.train), split_seed=2026, validation_fraction=0.2)


def validate_training_bundle(bundle_dir, split, expected_epochs=24):
    bundle_dir = Path(bundle_dir)
    config = _read_json(bundle_dir / "config.json")
    manifest = _read_json(bundle_dir / "training-manifest.json")
    if manifest.get("observations") != split["training"]:
        raise ValueError("bundle training observation manifest differs from fixed train558")
    if manifest.get("observation_count") != len(split["training"]):
        raise ValueError("bundle observation count mismatch")
    if config.get("epochs") != expected_epochs or manifest.get("epochs") != expected_epochs:
        raise ValueError("only the completed requested training bundle is accepted")
    history = manifest.get("history", [])
    if [row.get("epoch") for row in history] != list(range(1, expected_epochs + 1)):
        raise ValueError("bundle does not contain every completed training epoch")
    if any(not math.isfinite(row["mean_training_loss"]) for row in history):
        raise ValueError("nonfinite training history")
    return dict(weights_sha256=_hash_file(bundle_dir / "weights.pt"),
                config_sha256=_hash_file(bundle_dir / "config.json"),
                training_manifest_sha256=_hash_file(bundle_dir / "training-manifest.json"),
                training_summary_sha256=_digest(manifest),
                split_sha256=_digest(split), config=config, source_sha256=source_manifest())


def _gt_universe(index, observation):
    # get_overlap_df in organizer v6 enumerates GT CSV filament prefixes.
    # An annotator-image with no filament rows does not enter that table.
    included = [image_id for image_id in index.observations[observation]
                if len(index.segmentations[image_id]) > 0]
    skipped = [image_id for image_id in index.observations[observation] if image_id not in included]
    return included, skipped


def _comparison_counts(index, observation, predictions):
    comparisons, total = {}, _counts()
    included, skipped = _gt_universe(index, observation)
    for image_id in included:
        score = instance_pq(predictions, ground_truth_masks(index, image_id), expected_shape=OFFICIAL_SHAPE)
        comparisons[image_id] = score.to_dict()
        _add_counts(total, score)
    return comparisons, total, skipped


def _masks(labels):
    for number in range(1, int(labels.max()) + 1):
        yield labels == number


def evaluate_development(model, index, image_root, split, output_path):
    """Only used when no real completed dev receipt is available yet."""
    totals = {str(threshold): _counts() for threshold in THRESHOLDS}
    records = []
    for observation in split["development"]:
        image = read_observation(index, image_root, observation)
        probability = model.predict_proba(image)
        row = dict(observation=observation, thresholds={})
        for threshold in THRESHOLDS:
            labels = model.labels_from_probability(image, probability, threshold)
            comparisons, counts, skipped = _comparison_counts(index, observation, list(_masks(labels)))
            _add_counts(totals[str(threshold)], counts)
            row["thresholds"][str(threshold)] = dict(prediction_count=int(labels.max()),
                comparisons=comparisons, zero_gt_sets_excluded=skipped)
        records.append(row)
        print("FINAL_DEV", json.dumps({"observation": observation}), flush=True)
    report = dict(status="DEVELOPMENT_ONLY_TRAIN558_DEV8_DISJOINT_HOLDOUT141_UNTOUCHED",
                  config=model.config.to_dict(), training=split["training"],
                  training_summary=model.training_summary_,
                  development=split["development"], holdout_reserved=len(split["holdout"]),
                  trials=[dict(threshold=t, counts=totals[str(t)], official_global_pq=_pq(totals[str(t)])) for t in THRESHOLDS],
                  records=records)
    _json(output_path, report)
    return report


def _development_totals(report, split, index, source_config):
    if report.get("training") != split["training"] or report.get("development") != split["development"]:
        raise ValueError("development receipt has different train/dev observations")
    if report.get("holdout_reserved") != len(split["holdout"]):
        raise ValueError("development receipt holdout count differs")
    config = dict(report.get("config", {}))
    expected = dict(source_config)
    config.pop("device", None)
    expected.pop("device", None)
    # JSON represents tuples as lists; canonicalization treats both the same.
    if _canonical(config) != _canonical(expected):
        raise ValueError("development receipt model config differs from artifact")
    records = report.get("records", [])
    if [r.get("observation") for r in records] != split["development"]:
        raise ValueError("development receipt must contain exactly the fixed dev8 in order")
    totals = {str(threshold): _counts() for threshold in THRESHOLDS}
    skipped_ids = set()
    for row in records:
        image_ids, skipped = _gt_universe(index, row["observation"])
        skipped_ids.update(skipped)
        if set(row.get("thresholds", {})) != set(totals):
            raise ValueError("development receipt threshold set changed")
        for key in totals:
            comparisons = row["thresholds"][key].get("comparisons", {})
            if not set(image_ids) <= set(comparisons) or not set(comparisons) <= set(index.observations[row["observation"]]):
                raise ValueError("development receipt annotator-image membership differs")
            for image_id in image_ids:
                score = comparisons[image_id]
                if any(not isinstance(score[k], int) or isinstance(score[k], bool) or score[k] < 0 for k in ("tp", "fp", "fn")):
                    raise ValueError("invalid development integer counts")
                if not isinstance(score["sum_iou"], (int, float)) or not math.isfinite(score["sum_iou"]) or score["sum_iou"] < 0:
                    raise ValueError("invalid development overlap sum")
                _add_counts(totals[key], score)
    return totals, sorted(skipped_ids)


def freeze_threshold(output_dir, identity, split, report, index):
    """One deterministic dev-only choice; any incompatible existing freeze fails."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    if "training_summary_sha256" in identity and _digest(report.get("training_summary")) != identity["training_summary_sha256"]:
        raise ValueError("development receipt training summary differs from completed model bundle")
    totals, excluded = _development_totals(report, split, index, identity["config"])
    trials = [dict(threshold=t, counts=totals[str(t)], official_global_pq=_pq(totals[str(t)])) for t in THRESHOLDS]
    selected = max(trials, key=lambda row: row["official_global_pq"])
    payload = dict(schema_version=1, weights_sha256=identity["weights_sha256"],
        config_sha256=identity["config_sha256"], training_manifest_sha256=identity["training_manifest_sha256"],
        split_sha256=identity["split_sha256"], development_report_sha256=_digest(report),
        source_sha256=identity["source_sha256"],
        thresholds_predefined=list(THRESHOLDS), tie_policy="first in predefined order",
        selected_threshold=selected["threshold"], development_trials=trials,
        zero_gt_sets_excluded=excluded, selection_source="fixed dev8 only, before holdout/test")
    payload["freeze_id"] = _digest(payload)
    path = output_dir / "threshold-freeze.json"
    if path.exists():
        existing = _read_json(path)
        comparable = dict(existing)
        comparable.pop("created_at_utc", None)
        if comparable != payload:
            raise ValueError("existing threshold freeze is incompatible; no overwrite or re-tuning")
        return existing
    payload["created_at_utc"] = _now()
    _json(path, payload)
    return payload


def _journal(path, context_id, allowed_names):
    seen = {}
    if Path(path).exists():
        for line in Path(path).read_text(encoding="utf-8").splitlines():
            row = json.loads(line)
            name = row.get("observation")
            if row.get("context_id") != context_id or name not in allowed_names or name in seen:
                raise ValueError("incompatible or duplicate resume journal entry")
            seen[name] = row
    return seen


def _append(path, row):
    with Path(path).open("a", encoding="utf-8", newline="\n") as stream:
        stream.write(json.dumps(row, allow_nan=False) + "\n")
        stream.flush()


def _figure(model, image, labels, index, observation, counts, output_dir, display_figures):
    import matplotlib.pyplot as plt
    union = np.zeros(image.shape, dtype=bool)
    for image_id in index.observations[observation]:
        for mask in ground_truth_masks(index, image_id):
            union |= mask
    # Figures are derived comparison panels for the noncommercial report,
    # not a redistribution of original JPEGs or annotation files.
    size = (768, 768)
    gray = cv2.resize(image, size, interpolation=cv2.INTER_AREA)
    target = cv2.resize(union.astype(np.uint8), size, interpolation=cv2.INTER_NEAREST)
    predicted = cv2.resize((labels > 0).astype(np.uint8), size, interpolation=cv2.INTER_NEAREST)
    figure, axes = plt.subplots(1, 3, figsize=(12.5, 4.5), constrained_layout=True)
    for axis in axes:
        axis.imshow(gray, cmap="gray", vmin=0, vmax=255)
        axis.axis("off")
    axes[0].set_title("H-alpha observation")
    axes[1].set_title("GT union boundary (display only)")
    axes[2].set_title("Prediction union boundary")
    if target.any() and not target.all():
        axes[1].contour(target, levels=[0.5], colors=["#40dd70"], linewidths=0.65)
    if predicted.any() and not predicted.all():
        axes[2].contour(predicted, levels=[0.5], colors=["#ff9f31"], linewidths=0.65)
    figure.suptitle(observation + " | per-observation global PQ=" + format(_pq(counts), ".4f") +
                    " | TP=" + str(counts["tp"]) + " FP=" + str(counts["fp"]) + " FN=" + str(counts["fn"]), fontsize=10)
    figure.text(0.5, 0.005, "MAGFiLO / NSO GONG H-alpha — CC BY-NC 4.0. Fixed first-three holdout examples; GT annotators scored separately.",
                ha="center", fontsize=7)
    stem = "holdout-example-" + Path(observation).stem
    png, pdf = Path(output_dir) / (stem + ".png"), Path(output_dir) / (stem + ".pdf")
    figure.savefig(png, dpi=140, bbox_inches="tight")
    figure.savefig(pdf, bbox_inches="tight")
    if display_figures:
        from IPython.display import display
        display(figure)
    plt.close(figure)
    return dict(observation=observation, selection="first three fixed sorted holdout names, chosen before evaluation",
                per_observation_global_pq=_pq(counts), counts=counts,
                files=[dict(name=p.name, bytes=p.stat().st_size, sha256=_hash_file(p)) for p in (png, pdf)],
                ground_truth_union_is_display_only=True)


def bootstrap_observation_pq(records, resamples=1000, seed=2026):
    """Exploratory cluster bootstrap from existing counts, no new inference."""
    if not records or len({r["observation"] for r in records}) != len(records):
        raise ValueError("bootstrap requires unique observation records")
    if isinstance(resamples, bool) or not isinstance(resamples, int) or not 1 <= resamples <= 10000:
        raise ValueError("bootstrap resamples must be in 1..10000")
    values = np.asarray([[r["counts"][key] for key in ("sum_iou", "tp", "fp", "fn")]
                         for r in records], dtype=np.float64)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("bootstrap counts must be finite and nonnegative")
    rng = np.random.default_rng(seed)
    sampled = rng.integers(0, len(records), size=(resamples, len(records)))
    total = values[sampled].sum(axis=1)
    denominator = total[:, 1] + 0.5 * total[:, 2] + 0.5 * total[:, 3]
    scores = np.divide(total[:, 0], denominator, out=np.zeros(resamples), where=denominator > 0)
    lower, upper = np.percentile(scores, [2.5, 97.5])
    return dict(method="observation-cluster nonparametric percentile bootstrap",
        cluster="one physical observation with all annotator-image counts kept together",
        resamples=resamples, seed=seed, percentile_interval=[2.5, 97.5],
        interval_95=[float(lower), float(upper)], observation_count=len(records),
        exploratory=True, guaranteed_coverage=False, inference_rerun=False,
        limitations="Hash-based filename split does not establish temporal independence. Correlated observations and small development set limit interpretation; interval coverage is not guaranteed.")


def evaluate_holdout(model, index, image_root, split, freeze, output_dir, display_figures=True):
    output_dir = Path(output_dir)
    figures_dir = output_dir / "figures"
    figures_dir.mkdir(exist_ok=True)
    names = split["holdout"]
    chosen_figures = names[:3]  # Freeze before looking at any holdout predictions.
    journal = output_dir / "holdout-results.jsonl"
    completed = _journal(journal, freeze["freeze_id"], set(names))
    for number, observation in enumerate(names, start=1):
        if observation in completed:
            continue
        image = read_observation(index, image_root, observation)
        labels = model.predict_labels(image, freeze["selected_threshold"])
        comparisons, counts, excluded = _comparison_counts(index, observation, list(_masks(labels)))
        row = dict(context_id=freeze["freeze_id"], observation=observation,
                   prediction_count=int(labels.max()), comparisons=comparisons,
                   counts=counts, official_global_pq=_pq(counts), zero_gt_sets_excluded=excluded)
        if observation in chosen_figures:
            row["figure"] = _figure(model, image, labels, index, observation, counts, figures_dir, display_figures)
        _append(journal, row)
        completed[observation] = row
        if number <= 3 or number % 10 == 0 or number == len(names):
            print("FINAL_HOLDOUT_PROGRESS", json.dumps(dict(completed=number, total=len(names))), flush=True)
    totals = _counts()
    for observation in names:
        _add_counts(totals, completed[observation]["counts"])
    figures = [completed[name]["figure"] for name in chosen_figures]
    for item in figures:
        for entry in item["files"]:
            path = figures_dir / entry["name"]
            if not path.is_file() or path.stat().st_size != entry["bytes"] or _hash_file(path) != entry["sha256"]:
                raise ValueError("completed holdout figure is missing or changed")
    report = dict(status="HELDOUT_EVALUATION_COMPLETE_NO_RETUNING", context_id=freeze["freeze_id"],
        observation_count=len(names), observations=names, threshold=freeze["selected_threshold"],
        counts=totals, official_global_pq=_pq(totals), comparison_count=sum(len(completed[n]["comparisons"]) for n in names),
        zero_gt_sets_excluded=sorted({i for n in names for i in completed[n]["zero_gt_sets_excluded"]}),
        score_universe="Organizer v6 GT-filament-row universe; each nonempty annotator-image scored separately",
        uncertainty=bootstrap_observation_pq([completed[name] for name in names]),
        figures=figures, completed_at_utc=_now(), retuning_permitted=False)
    _json(output_dir / "holdout-results.json", report)
    _json(figures_dir / "figures-manifest.json", dict(attribution="MAGFiLO / NSO GONG H-alpha; CC BY-NC 4.0; noncommercial competition report",
        selection=chosen_figures, selection_before_holdout_evaluation=True, figures=figures))
    return report


def mask_digest(mask):
    return hashlib.sha256(np.packbits(np.asarray(mask, dtype=bool).ravel(), bitorder="big").tobytes()).hexdigest()


def validate_structural_rows(rows, valid_stems):
    seen = set()
    for row in rows:
        if set(row) != {"filament_id", "segmentation_rle"}:
            raise ValueError("CSV columns differ from official contract")
        filament_id, counts = row["filament_id"], row["segmentation_rle"]
        if not isinstance(filament_id, str) or "_" not in filament_id:
            raise ValueError("filament_id lacks a suffix")
        stem, suffix = filament_id.split("_", 1)
        if stem not in valid_stems or not suffix or filament_id in seen:
            raise ValueError("unknown image stem, empty suffix, or duplicate filament_id")
        if not isinstance(counts, str) or not counts or any(c in counts for c in ("'", '"', "\n", "\r")):
            raise ValueError("counts must be a nonempty bare compressed string")
        counts.encode("ascii")
        seen.add(filament_id)
    return seen


def discover_test_images(test_root, expected_count=180):
    root = Path(test_root).resolve()
    paths = sorted(root.glob("*.jpeg"))
    if len(paths) != expected_count or any(p.resolve().parent != root for p in paths):
        raise ValueError("unexpected test image count or unsafe path")
    stems = [path.stem for path in paths]
    if len(set(stems)) != len(stems) or any("_" in stem or not stem for stem in stems):
        raise ValueError("test stem cannot be parsed unambiguously by official first-underscore reader")
    return paths


def infer_test(model, test_root, freeze, output_dir, expected_count=180):
    from pycocotools import mask as mask_api  # Actual installed dependency; no fallback.
    output_dir = Path(output_dir)
    paths = discover_test_images(test_root, expected_count)
    valid_names, valid_stems = {p.name for p in paths}, {p.stem for p in paths}
    journal = output_dir / "test-predictions.jsonl"
    completed = _journal(journal, freeze["freeze_id"], valid_names)
    for number, path in enumerate(paths, start=1):
        if path.name in completed:
            continue
        image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        if image is None or image.shape != OFFICIAL_SHAPE or image.dtype != np.uint8:
            raise ValueError("test image differs from official grayscale shape/dtype")
        labels = model.predict_labels(image, freeze["selected_threshold"])
        mask_hashes = {}
        def masks():
            for label, mask in enumerate(_masks(labels), start=1):
                mask_hashes[path.stem + "_" + str(label)] = mask_digest(mask)
                yield mask
        rows = submission_rows(path.name, masks(), expected_shape=OFFICIAL_SHAPE, first_instance=1)
        validate_structural_rows(rows, valid_stems)
        row = dict(context_id=freeze["freeze_id"], observation=path.name,
                   prediction_count=len(rows), rows=rows, mask_sha256=mask_hashes,
                   native_coco_encode_decode_roundtrip=True,
                   zero_detection_policy=EMPTY_POLICY if not rows else None)
        _append(journal, row)
        completed[path.name] = row
        if number % 10 == 0 or number == len(paths):
            print("FINAL_TEST_PROGRESS", json.dumps(dict(completed=number, total=len(paths))), flush=True)
    rows = [row for p in paths for row in completed[p.name]["rows"]]
    all_ids = validate_structural_rows(rows, valid_stems)
    if sum(completed[p.name]["prediction_count"] for p in paths) != len(rows):
        raise ValueError("test manifest instance count mismatch")
    csv_path = output_dir / "submission.csv"
    with csv_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["filament_id", "segmentation_rle"])
        writer.writeheader()
        writer.writerows(rows)
    # Verify the actual written CSV and decode every row with native COCO.
    with csv_path.open(encoding="utf-8", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["filament_id", "segmentation_rle"]:
            raise ValueError("written CSV header differs")
        read_back = list(reader)
    if read_back != rows:
        raise ValueError("actual CSV serialization roundtrip changed row data")
    validate_structural_rows(read_back, valid_stems)
    for row in read_back:
        stem = row["filament_id"].split("_", 1)[0]
        decoded = mask_api.decode(dict(size=list(OFFICIAL_SHAPE), counts=row["segmentation_rle"].encode("ascii")))
        if decoded.shape != OFFICIAL_SHAPE or not decoded.any():
            raise ValueError("actual CSV RLE decoded to wrong shape or empty foreground")
        expected = completed[stem + ".jpeg"]["mask_sha256"][row["filament_id"]]
        if mask_digest(decoded) != expected:
            raise ValueError("actual CSV RLE differs from original inferred mask")
    zero_names = [p.name for p in paths if not completed[p.name]["rows"]]
    manifest = dict(status="LOCAL_CSV_COMPLETE_NOT_UPLOADED_OR_SERVER_VALIDATED",
        context_id=freeze["freeze_id"], threshold=freeze["selected_threshold"],
        test_images_processed=len(paths), images_with_predictions=len(paths) - len(zero_names),
        zero_detection_images=zero_names, zero_detection_policy=EMPTY_POLICY,
        per_image_prediction_counts={p.name: completed[p.name]["prediction_count"] for p in paths},
        row_count=len(rows), unique_row_ids=len(all_ids), original_mask_shape=list(OFFICIAL_SHAPE),
        actual_csv_reread_equal=True, native_coco_all_rows_roundtripped=True,
        native_coco_roundtripped_mask_count=len(rows),
        csv=dict(name=csv_path.name, bytes=csv_path.stat().st_size, sha256=_hash_file(csv_path)),
        server_uploaded=False, server_acceptance_verified=False, completed_at_utc=_now())
    _json(output_dir / "test-coverage-manifest.json", manifest)
    return manifest


def source_manifest():
    paths = {name + ".py": Path(sys.modules[name].__file__)
             for name in ("dataset", "evaluation", "unet_candidate")}
    paths["pipeline.py"] = Path(__file__)
    return {name: _hash_file(path) for name, path in paths.items()}


def create_delivery_zip(bundle_dir, output_dir, archive_path):
    """Package only explicit own model/results files; never recurse over data."""
    bundle_input, output_input, archive_input = map(Path, (bundle_dir, output_dir, archive_path))
    if any(path.is_symlink() for path in (bundle_input, output_input, archive_input)):
        raise ValueError("delivery roots and archive must not be symlinks")
    bundle_root, output_root = bundle_input.resolve(), output_input.resolve()
    archive = archive_input.resolve()
    if bundle_root == output_root or bundle_root in output_root.parents or output_root in bundle_root.parents:
        raise ValueError("model and output roots must be separate")
    if any(archive == root or root in archive.parents for root in (bundle_root, output_root)):
        raise ValueError("archive must be outside both input roots")
    if archive.suffix.lower() != ".zip":
        raise ValueError("delivery archive must have .zip extension")
    completion = _read_json(output_root / "completion.json")
    if completion.get("status") != "LOCAL_PIPELINE_COMPLETE_NO_OFFICIAL_SUBMISSION":
        raise ValueError("only a completed local pipeline can be packaged")
    files = []
    def include(root, relative, prefix):
        relative = Path(relative)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("unsafe delivery relative filename")
        source = root / relative
        if any(part.is_symlink() for part in (source, *source.parents) if part != root.parent):
            raise ValueError("delivery input must not traverse symlinks")
        resolved = source.resolve()
        if root not in resolved.parents or not source.is_file():
            raise ValueError("delivery file is missing or outside its explicit root")
        files.append((source, prefix + "/" + relative.as_posix()))
    for filename in ("weights.pt", "config.json", "training-manifest.json"):
        include(bundle_root, filename, "model")
    output_files = ("completion.json", "development-results.json", "threshold-freeze.json",
        "holdout-results.json", "holdout-results.jsonl", "test-predictions.jsonl", "submission.csv",
        "test-coverage-manifest.json", "runtime-manifest.json", "split-manifest.json", "zero-gt-universe.json")
    for filename in output_files:
        include(output_root, filename, "outputs")
    include(output_root, "figures/figures-manifest.json", "outputs")
    figures = _read_json(output_root / "figures" / "figures-manifest.json")
    for figure in figures["figures"]:
        for entry in figure["files"]:
            filename = entry["name"]
            if Path(filename).name != filename or Path(filename).suffix not in (".png", ".pdf"):
                raise ValueError("only named derived PNG/PDF report figures can be packaged")
            include(output_root, Path("figures") / filename, "outputs")
            source = output_root / "figures" / filename
            if source.stat().st_size != entry["bytes"] or _hash_file(source) != entry["sha256"]:
                raise ValueError("derived report figure differs from completed manifest")
    names = [name for _, name in files]
    if len(names) != len(set(names)):
        raise ValueError("duplicate delivery entry")
    manifest = dict(schema_version=1, files=[dict(name=name, bytes=source.stat().st_size,
        sha256=_hash_file(source)) for source, name in files], source_sha256=source_manifest(),
        freeze_id=completion["freeze_id"], raw_dataset_included=False,
        official_submission_completed=False, created_at_utc=_now(),
        figure_attribution="MAGFiLO / NSO GONG H-alpha; CC BY-NC 4.0; noncommercial competition report",
        packaging_policy="Explicit model and result allowlist only; no input images, annotations, credentials or private audits")
    archive.parent.mkdir(parents=True, exist_ok=True)
    temporary = archive.with_suffix(".zip.partial")
    if temporary.is_symlink():
        raise ValueError("temporary archive must not be a symlink")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as writer:
        for source, name in files:
            writer.write(source, arcname=name)
        writer.writestr("delivery-manifest.json", json.dumps(manifest, indent=2, allow_nan=False) + "\n")
    with zipfile.ZipFile(temporary) as reader:
        if reader.namelist() != names + ["delivery-manifest.json"] or reader.testzip() is not None:
            raise ValueError("delivery archive integrity check failed")
    temporary.replace(archive)
    receipt = dict(path=str(archive), bytes=archive.stat().st_size, sha256=_hash_file(archive),
                   file_count=len(files) + 1, raw_dataset_included=False, official_submission_completed=False)
    print("FINAL_DELIVERY_ZIP", json.dumps(receipt), flush=True)
    return receipt


def runtime_manifest():
    import torch
    distributions = {}
    for name in ("numpy", "scipy", "opencv-python", "opencv-python-headless", "pandas", "torch", "torchvision",
                 "pycocotools", "matplotlib", "Pillow", "joblib", "scikit-learn", "ipython"):
        try:
            distributions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            distributions[name] = None
    return dict(python=sys.version, platform=platform.platform(), packages=distributions,
                opencv_runtime_version=cv2.__version__, opencv_build_information=cv2.getBuildInformation(),
                torch_cuda_build=torch.version.cuda, cuda_available=torch.cuda.is_available(),
                cuda_devices=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
                source_sha256=source_manifest())


def run_posttraining(annotation_path, image_root, test_root, bundle_dir, output_dir,
                     development_report_path=None, device="cuda", display_figures=True):
    """Resume only: does not call fit, access networks, or upload anything."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    index = load_coco_index(annotation_path)
    split = partitions(index)
    identity = validate_training_bundle(bundle_dir, split)
    model = UNetCandidate.load_artifacts(bundle_dir, device=device)
    _json(output_dir / "runtime-manifest.json", runtime_manifest())
    _json(output_dir / "split-manifest.json", split)
    zero_gt = sorted(i for i in index.images if not index.segmentations[i])
    _json(output_dir / "zero-gt-universe.json", dict(zero_gt_annotator_image_ids=zero_gt,
        count=len(zero_gt), evaluation_policy="exclude from organizer GT-filament-row universe; record rather than merge annotators"))
    copied_development = output_dir / "development-results.json"
    if development_report_path is not None:
        report = _read_json(development_report_path)
        if copied_development.exists() and _digest(_read_json(copied_development)) != _digest(report):
            raise ValueError("a different development receipt already exists; no replacement")
        if not copied_development.exists():
            _json(copied_development, report)
    elif copied_development.exists():
        report = _read_json(copied_development)
    else:
        report = evaluate_development(model, index, image_root, split, copied_development)
    freeze = freeze_threshold(output_dir, identity, split, report, index)
    print("FINAL_THRESHOLD_FROZEN", json.dumps(freeze), flush=True)
    holdout = evaluate_holdout(model, index, image_root, split, freeze, output_dir, display_figures)
    print("FINAL_HOLDOUT_SUMMARY", json.dumps(holdout), flush=True)
    test = infer_test(model, test_root, freeze, output_dir)
    _json(output_dir / "completion.json", dict(status="LOCAL_PIPELINE_COMPLETE_NO_OFFICIAL_SUBMISSION",
        freeze_id=freeze["freeze_id"], threshold=freeze["selected_threshold"],
        holdout_official_global_pq=holdout["official_global_pq"], holdout_observations=holdout["observation_count"],
        test_images_processed=test["test_images_processed"], csv=test["csv"],
        zero_detection_images=test["zero_detection_images"], official_submission_completed=False,
        completed_at_utc=_now(), train_annotation_sha256=_hash_file(annotation_path)))
    print("FINAL_PIPELINE_COMPLETE", json.dumps(_read_json(output_dir / "completion.json")), flush=True)
    return dict(freeze=freeze, holdout=holdout, test=test)


def train_from_scratch(annotation_path, image_root, bundle_dir, checkpoint_dir, device="cuda"):
    """Explicit optional reproduction; never called by the resume runner."""
    index = load_coco_index(annotation_path)
    split = partitions(index)
    config = UNetConfig(device=device, epochs=24, batch_size=4, min_area=300)
    model = UNetCandidate(config)
    model.fit_coco(index, image_root, split["training"],
        allowed_training_observations=split["combined_train"],
        forbidden_observations=tuple(split["holdout"]) + tuple(split["development"]),
        progress=lambda row: print("UNET_EPOCH", json.dumps(row), flush=True), checkpoint_dir=checkpoint_dir)
    model.save_artifacts(bundle_dir)
    return model


## 3. Verify source and inspect the split


In [ ]:
import hashlib, importlib, json, sys
expected_source_sha256 = {'dataset.py': '0fb04691dfd78dbb121eb56362100a83616ca4c174169a3a0f18b3d07346321c', 'evaluation.py': '4548a68482a2401fd6ee4b73a51f21eef07f8ae477551c3b4d9d8a4f3b64df67', 'unet_candidate.py': '04de330d8b304775487ea35216d87ee422391f5b8a4eca972dd15aa032cc8fe0', 'pipeline.py': '63527af6fa04e722e0e7fd6af22f8eb244b819722a87ac607250189f9e5b1996'}
for filename, expected in expected_source_sha256.items():
    actual = hashlib.sha256((SOURCE_ROOT / filename).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError("Source hash mismatch: " + filename)
sys.path.insert(0, str(SOURCE_ROOT))
for module_name in ("pipeline", "unet_candidate", "dataset", "evaluation"):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
from pipeline import run_posttraining, train_from_scratch, partitions, create_delivery_zip
from dataset import load_coco_index
preview_split = partitions(load_coco_index(ANNOTATION_PATH))
print(json.dumps({name: len(preview_split[name]) for name in ("training", "development", "holdout")}))


## 4. Optional from-scratch reproduction
Resume mode skips this cell's training branch. The original implementation uses no pretrained weights. GPU seeds are fixed but cross-platform bitwise determinism is not promised.


In [ ]:
if MODE == "train":
    train_from_scratch(ANNOTATION_PATH, IMAGE_ROOT, MODEL_DIR, CHECKPOINT_DIR, DEVICE)
elif MODE != "resume":
    raise ValueError("MODE must be resume or train")


## 5. Dev freeze, untouched holdout, figures and test CSV
The completed development receipt is reused when available; it is not recomputed. A compatible existing freeze and journals resume prior work. A changed model, split, source, or development receipt fails instead of replacing the freeze. The first three sorted holdout observations were chosen before scoring.


In [ ]:
final_results = run_posttraining(
    ANNOTATION_PATH, IMAGE_ROOT, TEST_ROOT, MODEL_DIR, OUTPUT_DIR,
    development_report_path=DEVELOPMENT_REPORT if DEVELOPMENT_REPORT.is_file() else None,
    device=DEVICE, display_figures=True,
)
delivery_receipt = create_delivery_zip(MODEL_DIR, OUTPUT_DIR, DELIVERY_ZIP)


## 6. Local output checklist
`threshold-freeze.json`, `holdout-results.json`, `figures/`, `test-coverage-manifest.json`, `runtime-manifest.json`, `split-manifest.json`, and `submission.csv` record the actual run. All CSV rows are reread and decoded with native pycocotools and compared with hashes of the original inferred masks. No-detection images appear in coverage rather than as fabricated filaments. The printed delivery ZIP contains only the three model files and explicit result files, plus a per-file hash manifest. It excludes original images and annotations and can be downloaded through the normal notebook output UI.


In [ ]:
for filename in ("completion.json", "test-coverage-manifest.json", "runtime-manifest.json"):
    print(filename)
    print((OUTPUT_DIR / filename).read_text(encoding="utf-8"))
